# Dataset Self-Conformance Demo

Every bundled dataset should satisfy the same basic invariants:

1. Connectivity indices point to nodes that actually exist.
2. The declared basis is one `pyfieldml` recognises.
3. Evaluating the coordinate field at the reference-element centroid returns a finite point.

This notebook runs those checks over every dataset as a fail-fast sanity screen, then **visualizes** the structural invariants across the zoo so regressions jump out at a glance.

In [ ]:
import numpy as np

from pyfieldml import datasets
from pyfieldml.interop.meshio import _basis_topology_order, _find_basis_name

## Reference-element centroids

Each supported topology has its own centroid in xi-space:

In [ ]:
CENTROIDS = {
    "line": (0.5,),
    "triangle": (1.0 / 3.0, 1.0 / 3.0),
    "quad": (0.5, 0.5),
    "tet": (0.25, 0.25, 0.25),
    "hex": (0.5, 0.5, 0.5),
    "wedge": (1.0 / 3.0, 1.0 / 3.0, 0.5),
}

RECOGNIZED_TOPOLOGIES = set(CENTROIDS)

## Run the invariants over every bundled dataset

In [ ]:
def check(name):
    doc = datasets.load(name)
    coords = doc.evaluators["coordinates"].as_ndarray()
    conn = doc.evaluators["coordinates.connectivity"].as_ndarray().astype(np.int64)
    n_nodes = coords.shape[0]

    # (1) connectivity stays within node count (1-indexed)
    assert conn.min() >= 1, f"{name}: connectivity has index < 1"
    assert conn.max() <= n_nodes, (
        f"{name}: connectivity max {conn.max()} exceeds node count {n_nodes}"
    )

    # (2) basis is recognized
    basis = _find_basis_name(doc.region)
    topology, order = _basis_topology_order(basis)
    assert topology in RECOGNIZED_TOPOLOGIES, f"{name}: bad topology {topology!r}"

    # (3) evaluating at the element centroid returns finite numbers
    field = doc.field("coordinates")
    xi = CENTROIDS[topology]
    centroid = field.evaluate(element=1, xi=xi)
    assert np.all(np.isfinite(centroid)), f"{name}: non-finite centroid"
    return {
        "dataset": name,
        "n_nodes": n_nodes,
        "n_elems": conn.shape[0],
        "topology": topology,
        "order": order,
        "centroid": centroid,
        "coords": coords,
        "conn": conn,
    }


rows = [check(n) for n in datasets.list()]

## Summary table

In [ ]:
fmt = "{:22s} {:>7s} {:>7s} {:10s} {:>5s}  {}"
row_fmt = "{:22s} {:7d} {:7d} {:10s} {:5d}  {}"
header = fmt.format("dataset", "nodes", "elems", "topology", "order", "centroid")
print(header)
print("-" * len(header))
for row in rows:
    c = np.array2string(row["centroid"], precision=4, suppress_small=True)
    print(
        row_fmt.format(
            row["dataset"],
            row["n_nodes"],
            row["n_elems"],
            row["topology"],
            row["order"],
            c,
        )
    )
print()
print(f"all {len(rows)} bundled datasets pass self-conformance.")

## Visual invariant 1: bounding-box extents

BP3D meshes are stored in millimeters; synthetic primitives live in the unit interval. Plotting the per-axis extent on a log scale makes the unit conventions obvious.

In [ ]:
import matplotlib.pyplot as plt

names = [r["dataset"] for r in rows]
extents = np.array(
    [np.ptp(r["coords"], axis=0) for r in rows],
    dtype=np.float64,
)

fig, ax = plt.subplots(figsize=(10, 3.8))
x = np.arange(len(names))
w = 0.25
axis_colors = ["#4C78A8", "#F58518", "#54A24B"]
for i, (ax_label, color) in enumerate(zip("xyz", axis_colors, strict=True)):
    ax.bar(x + (i - 1) * w, extents[:, i], width=w, color=color, label=f"extent_{ax_label}")
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha="right")
ax.set_ylabel("axis extent (log)")
ax.set_title("Bounding-box extents across the model zoo")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()

## Visual invariant 2: connectivity-degree histograms

For each dataset, count how many elements touch each node. Manifold surfaces peak around 6; pathological or heavily joined meshes (e.g., the 74-component skull) skew the distribution.

In [ ]:
n = len(rows)
cols = 4
rows_ = (n + cols - 1) // cols
fig, axes = plt.subplots(rows_, cols, figsize=(11, 2.4 * rows_))
axes = axes.flatten()
for ax, r in zip(axes, rows, strict=False):
    flat = r["conn"].ravel()
    counts = np.bincount(flat, minlength=r["n_nodes"] + 1)[1:]
    ax.hist(counts, bins=range(0, int(counts.max()) + 2), color="steelblue", edgecolor="black")
    ax.set_title(f"{r['dataset']}\n(max deg {int(counts.max())})", fontsize=9)
    ax.set_xlabel("element-incidence per node")
    ax.tick_params(labelsize=7)
for ax in axes[len(rows) :]:
    ax.set_visible(False)
fig.tight_layout()

## Visual invariant 3: one element per dataset, colored by topology

Extract the *first* element from each dataset and render it solo in a grid. Topology differences (hex vs tet vs triangle) are visually unambiguous.

In [ ]:
import pyvista as pv

pv.OFF_SCREEN = True
pv.set_jupyter_backend("static")

TOPO_COLORS = {
    "hex": "#4C78A8",
    "tet": "#F58518",
    "triangle": "#54A24B",
    "quad": "#E45756",
    "wedge": "#B279A2",
    "line": "#9D755D",
}

cols = 5
nrows = (len(rows) + cols - 1) // cols
p = pv.Plotter(
    off_screen=True,
    window_size=(1500, 320 * nrows),
    shape=(nrows, cols),
)
for i, r in enumerate(rows):
    p.subplot(i // cols, i % cols)
    first_conn = r["conn"][0] - 1  # 1-indexed -> 0-indexed
    first_pts = r["coords"][first_conn]
    cloud = pv.PolyData(first_pts)
    p.add_mesh(
        cloud,
        color=TOPO_COLORS.get(r["topology"], "gray"),
        render_points_as_spheres=True,
        point_size=18,
    )
    # draw edges of the first element
    n_v = first_pts.shape[0]
    lines = []
    for a in range(n_v):
        for b in range(a + 1, n_v):
            lines += [2, a, b]
    edges = pv.PolyData(first_pts, lines=np.array(lines, dtype=np.int64))
    p.add_mesh(edges, color="black", line_width=2)
    p.add_text(
        f"{r['dataset']}\n{r['topology']}/order {r['order']}",
        font_size=8,
        position="upper_edge",
    )
    p.view_isometric()
p.show(screenshot="/tmp/conformance_first_elem.png")

### Why this matters

Bundled assets are the foundation for every downstream tutorial, benchmark, and user demo. This notebook is a lightweight smoke test you can run after any dataset edit to confirm nothing has silently corrupted. For a strict byte-level regression canary, see `tests/benchmarks/test_reproducibility.py`.